# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow python-dotenv
```

In [3]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [5]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A
FEATURE_DIR: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Put credentials in `HW02/HW02_A/.env` (see `.env.example`).
- Do not commit real passwords to Git (`.env` is in `.gitignore`).

In [6]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "")
DB_PORT = int(os.getenv("PGPORT", ""))
DB_NAME = os.getenv("PGDATABASE", "")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

Connecting to:
HOST: 185.50.38.163
PORT: 32112
DB: qbc12_airbnb
USER: student_samin_kakaei


{'database': 'qbc12_airbnb',
 'user_name': 'student_samin_kakaei',
 'server_ip': '172.19.0.2',
 'server_port': 5432,
 'checked_at': datetime.datetime(2026, 6, 3, 17, 23, 3, 171693, tzinfo=datetime.timezone.utc)}

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [7]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [8]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable
0,core,calendar_day,1,listing_id,bigint,NO
1,core,calendar_day,2,date,date,NO
2,core,calendar_day,3,available,boolean,YES
3,core,calendar_day,4,price,numeric,YES
4,core,calendar_day,5,adjusted_price,numeric,YES
5,core,calendar_day,6,minimum_nights,integer,YES
6,core,calendar_day,7,maximum_nights,integer,YES
7,core,host,1,host_id,bigint,NO
8,core,host,2,host_pseudo_id,text,NO
9,core,host,3,is_superhost,boolean,YES


In [9]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [10]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [12]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 10;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684
5,49552,225987,2,Entire home/apt,Entire guest suite,3,2.0,2.0,1 bath,322.0,3,1125,False,0363 576A D827 5085 6B83
6,50263,230246,1,Entire home/apt,Entire condo,4,2.0,3.0,1.5 baths,457.0,2,14,False,0363 7F3D 0BAE 28C8 C7D2
7,50515,231864,14,Entire home/apt,Entire townhouse,5,3.0,3.0,1.5 baths,198.0,7,18,False,0363 5DDB E495 A6D5 CEC6
8,50523,231946,2,Entire home/apt,Entire condo,2,1.0,1.0,1 bath,162.0,2,365,False,0363 22DC 0E52 B70B 0FB8
9,53921,252245,10,Entire home/apt,Entire rental unit,3,1.0,NaN,1 bath,NaN,1,21,False,0363 B43C B1D4 2666 3739



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False
5,443406859,cfe69dd56cffef559069b30216f8954b57e1de886b92a3...,False
6,2626085,43c4cdd7f1413364dd809c6c92c20baed35faa51f84e76...,False
7,74999205,1033d2369452b0969c1f63061af401f4581a92873b5659...,True
8,7969106,1b41831025e74b04a73fe88e99233d09f39660eac85bf6...,False
9,6127483,8cffc835ea5e24b102cf446e9d369edad9f3a2bf91bfbc...,True



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart
5,6,Bijlmer-Centrum
6,7,Geuzenveld - Slotermeer
7,8,Buitenveldert - Zuidas
8,9,Noord-West
9,10,IJburg - Zeeburgereiland



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,531281017,18062995,2019-09-17,73925523,2408ff6b678c0bdc30857dc6631d8762b57159ef9741c3...,392
1,531788922,18062995,2019-09-18,82572265,fb2e9dd6e8225de8231d388c9b9cf92b9c88f7e0823389...,13
2,532695264,18062995,2019-09-20,100595030,2503cafbac72a2a0a211282e2ed6e97c60058769ab3cf8...,467
3,537292491,18062995,2019-09-28,282542584,2612adccbea98416289b033feb314e33a3947244104fcf...,78
4,540324583,18062995,2019-10-03,116115856,7f6c33a5339d594d253c1aa0c9d5a2ae714321907af14c...,220
5,540880279,18062995,2019-10-04,65938182,b280baa9d72baf439c77b54c6ee11a42e4a9403fccec35...,145
6,542086221,18062995,2019-10-06,155392872,964cb3a74d1ccaaadd95915b842090390eed7754785aa5...,270
7,543734173,18062995,2019-10-08,52878420,5f477eaea2d8cd299828be54f31613eabe9e17072ff0bc...,204
8,544447121,18062995,2019-10-10,120972473,1314f5367fdcd67e16c7fb0828dad3be64be054d8042c2...,92
9,547826338,18062995,2019-10-16,31002341,6eedc8878c30f4380cbc920fd5ed63ff122aeba8f3bdf6...,91



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,857771032141263073,2026-06-20,False,None,None,2,45
1,857771032141263073,2026-06-21,False,None,None,2,45
2,857771032141263073,2026-06-22,False,None,None,2,45
3,857771032141263073,2026-06-23,False,None,None,2,45
4,857771032141263073,2026-06-24,False,None,None,2,45
5,857771032141263073,2026-06-25,False,None,None,2,45
6,857771032141263073,2026-06-26,False,None,None,2,45
7,857771032141263073,2026-06-27,False,None,None,2,45
8,857771032141263073,2026-06-28,False,None,None,2,45
9,857771032141263073,2026-06-29,False,None,None,2,45


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [13]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [14]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

# Earliest cutoff: we need PAST_WINDOW_DAYS of calendar history before cutoff.
earliest_cutoff_allowed_by_calendar = calendar_min_date + timedelta(days=PAST_WINDOW_DAYS)

# Latest cutoff: we need FUTURE_WINDOW_DAYS of calendar data after cutoff.
latest_cutoff_allowed_by_calendar = calendar_max_date - timedelta(days=FUTURE_WINDOW_DAYS)

# Pick the latest valid cutoff to use as much future label data as possible.
cutoff_date = latest_cutoff_allowed_by_calendar

# Feature window: [history_start_date, cutoff_date]
history_start_date = cutoff_date - timedelta(days=PAST_WINDOW_DAYS)

# Label window: (cutoff_date, label_end_date]
label_end_date = cutoff_date + timedelta(days=FUTURE_WINDOW_DAYS)

print("Calendar range:", calendar_min_date, "->", calendar_max_date)
print("Earliest allowed cutoff:", earliest_cutoff_allowed_by_calendar)
print("Latest allowed cutoff:", latest_cutoff_allowed_by_calendar)
print("Chosen cutoff_date:", cutoff_date)
print("Feature window:", history_start_date, "->", cutoff_date, f"({PAST_WINDOW_DAYS} days)")
print("Label window:", cutoff_date, "->", label_end_date, f"({FUTURE_WINDOW_DAYS} days, exclusive start)")

assert earliest_cutoff_allowed_by_calendar <= cutoff_date <= latest_cutoff_allowed_by_calendar
assert history_start_date >= calendar_min_date
assert label_end_date <= calendar_max_date
assert history_start_date <= cutoff_date <= label_end_date

Calendar range: 2025-09-11 -> 2026-09-10
Earliest allowed cutoff: 2025-12-10
Latest allowed cutoff: 2026-08-11
Chosen cutoff_date: 2026-08-11
Feature window: 2026-05-13 -> 2026-08-11 (90 days)
Label window: 2026-08-11 -> 2026-09-10 (30 days, exclusive start)


## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [15]:
# Document every sensitive or identity-linking column and what we do with it.
# This table is saved later as pii_audit_<version>.csv for reproducibility.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    {
        "table": "listing",
        "column": "host_id",
        "pii_type": "privacy-sensitive direct identifier",
        "decision": "use for join only; exclude from final features",
        "reason": "links listing to a specific host; join key only, not a model feature",
    },
    {
        "table": "host",
        "column": "host_id",
        "pii_type": "privacy-sensitive direct identifier",
        "decision": "use for join only; exclude from final features",
        "reason": "primary host key kept only during ETL joins; not a model feature",
    },
    {
        "table": "host",
        "column": "host_pseudo_id",
        "pii_type": "privacy-sensitive pseudonymized identifier",
        "decision": "exclude from final features",
        "reason": "still links records to the same host across tables; not a model feature",
    },
    {
        "table": "listing",
        "column": "license",
        "pii_type": "sensitive text",
        "decision": "exclude from final features",
        "reason": "government registration number that can identify a specific property or operator",
    },
    {
        "table": "listing",
        "column": "bathrooms_text",
        "pii_type": "raw text field",
        "decision": "exclude from final features",
        "reason": "parsed into numeric bathrooms; raw text is not needed in the ML dataset",
    },
    {
        "table": "listing",
        "column": "neighbourhood_id",
        "pii_type": "location surrogate key",
        "decision": "exclude from final features",
        "reason": "replaced by neighbourhood_name after join; ID alone is not a model feature",
    },
    {
        "table": "review",
        "column": "review_id",
        "pii_type": "privacy-sensitive direct identifier",
        "decision": "exclude; aggregate in SQL only",
        "reason": "review rows are aggregated before the ML dataset is built",
    },
    {
        "table": "review",
        "column": "reviewer_id",
        "pii_type": "privacy-sensitive direct identifier",
        "decision": "exclude; aggregate in SQL only",
        "reason": "identifies the guest who wrote a review; not a model feature",
    },
    {
        "table": "review",
        "column": "reviewer_pseudo_id",
        "pii_type": "privacy-sensitive pseudonymized identifier",
        "decision": "exclude; aggregate in SQL only",
        "reason": "still links reviews to the same guest; not a model feature",
    },
])

pii_audit

,table,column,pii_type,decision,reason
0,listing,listing_id,entity identifier,keep as entity key only,needed to define one row per listing; not a mo...
1,listing,host_id,privacy-sensitive direct identifier,use for join only; exclude from final features,links listing to a specific host; join key onl...
2,host,host_id,privacy-sensitive direct identifier,use for join only; exclude from final features,primary host key kept only during ETL joins; n...
3,host,host_pseudo_id,privacy-sensitive pseudonymized identifier,exclude from final features,still links records to the same host across ta...
4,listing,license,sensitive text,exclude from final features,government registration number that can identi...
5,listing,bathrooms_text,raw text field,exclude from final features,parsed into numeric bathrooms; raw text is not...
6,listing,neighbourhood_id,location surrogate key,exclude from final features,replaced by neighbourhood_name after join; ID ...
7,review,review_id,privacy-sensitive direct identifier,exclude; aggregate in SQL only,review rows are aggregated before the ML datas...
8,review,reviewer_id,privacy-sensitive direct identifier,exclude; aggregate in SQL only,identifies the guest who wrote a review; not a...
9,review,reviewer_pseudo_id,privacy-sensitive pseudonymized identifier,exclude; aggregate in SQL only,still links reviews to the same guest; not a m...


## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [20]:
# SQL to load the required listing columns.
listing_df = read_sql("""
SELECT
    listing_id,
    host_id,
    neighbourhood_id,
    room_type,
    property_type,
    accommodates,
    bedrooms,
    beds,
    bathrooms_text,
    listing_price,
    minimum_nights,
    maximum_nights,
    instant_bookable
FROM core.listing;
""")


# SQL to load the required host columns.
host_df = read_sql("""
SELECT
    host_id,
    is_superhost
FROM core.host;
""")


# SQL to load the neighbourhood columns.
neighbourhood_df = read_sql("""
SELECT
    neighbourhood_id,
    name AS neighbourhood_name
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

listing: (10480, 13)
host: (9201, 2)
neighbourhood: (22, 2)


## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [21]:
# normalize boolean columns.
# Example target columns:
# - listing_df["instant_bookable"]
# - host_df["is_superhost"]
listing_df["instant_bookable"] = listing_df["instant_bookable"].astype("boolean")
host_df["is_superhost"] = host_df["is_superhost"].astype("boolean")

# normalize numeric listing columns.
numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    listing_df[col] = pd.to_numeric(listing_df[col], errors="coerce")


def parse_bathrooms(text_value):
    """
    Convert bathrooms_text into a number.

    Examples:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - 'Half-bath' -> 0.5
    - missing/unrecognized -> NaN
    """
    # implement parser.
    if pd.isna(text_value):
        return np.nan

    text = str(text_value).strip().lower()
    if not text:
        return np.nan

    if "half-bath" in text or "half bath" in text:
        return 0.5

    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))

    return np.nan


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)

listing_df[["bathrooms_text", "bathrooms"]].head(10)

,bathrooms_text,bathrooms
0,1.5 baths,1.5
1,1 shared bath,1.0
2,1 shared bath,1.0
3,1.5 baths,1.5
4,1.5 baths,1.5
5,1 bath,1.0
6,1.5 baths,1.5
7,1.5 baths,1.5
8,1 bath,1.0
9,1 bath,1.0


## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [22]:
# create host_listing_features with one row per host_id.
host_listing_features = (
    listing_df.groupby("host_id", as_index=False)
    .agg(host_listing_count=("listing_id", "count"))
)

# merge listing, host, host_listing_features, and neighbourhood.
base_listing_features = (
    listing_df
    .merge(host_df, on="host_id", how="left")
    .merge(host_listing_features, on="host_id", how="left")
    .merge(neighbourhood_df, on="neighbourhood_id", how="left")
)

# choose privacy-safe static feature columns.
static_feature_cols = [
    "listing_id",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
    "instant_bookable",
    "is_superhost",
    "neighbourhood_name",
    "host_listing_count",
]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()

(10480, 14)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,instant_bookable,is_superhost,neighbourhood_name,host_listing_count
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,3,356,False,True,Centrum-West,1
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,89.0,2,730,False,True,Centrum-West,2
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,61.0,2,730,False,True,Centrum-Oost,2
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,730,False,False,Centrum-Oost,1
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,1125,False,False,Buitenveldert - Zuidas,1


## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [23]:
# write SQL aggregation over core.review.
# Use CAST(:cutoff_date AS date) when using the cutoff inside SQL.
review_features = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS total_reviews_before_cutoff,
        COUNT(DISTINCT reviewer_id) AS unique_reviewers_before_cutoff,
        AVG(comment_len) AS avg_comment_len_before_cutoff,
        MAX(comment_len) AS max_comment_len_before_cutoff,
        (CAST(:cutoff_date AS date) - MAX(review_date)) AS days_since_last_review
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

# convert feature columns to numeric where needed.
review_numeric_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
    "avg_comment_len_before_cutoff",
    "max_comment_len_before_cutoff",
    "days_since_last_review",
]

for col in review_numeric_cols:
    review_features[col] = pd.to_numeric(review_features[col], errors="coerce")

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()

(9383, 6)


,listing_id,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review
0,27886,311,311,302.167203,1917,338
1,28871,732,729,201.236339,1265,338
2,29051,849,841,245.108363,2253,337
3,44391,42,42,242.309524,891,1452
4,48373,5,5,272.200000,949,835


## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [24]:
# write SQL aggregation over core.calendar_day.
# Use history_start_date and cutoff_date.
#
# Required output examples:
# - available_days_last_90d
# - available_rate_last_90d
# - avg_minimum_nights_calendar_last_90d
# - avg_maximum_nights_calendar_last_90d
# - available_days_last_30d
# - available_rate_last_30d
# - avg_minimum_nights_calendar_last_30d
# - avg_maximum_nights_calendar_last_30d
calendar_features_all = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) FILTER (WHERE available IS TRUE) AS available_days_last_90d,
        COUNT(*) FILTER (WHERE available IS TRUE)::double precision
            / NULLIF(COUNT(*), 0) AS available_rate_last_90d,
        AVG(minimum_nights) AS avg_minimum_nights_calendar_last_90d,
        AVG(maximum_nights) AS avg_maximum_nights_calendar_last_90d,
        COUNT(*) FILTER (
            WHERE available IS TRUE
              AND date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        ) AS available_days_last_30d,
        COUNT(*) FILTER (
            WHERE available IS TRUE
              AND date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        )::double precision
            / NULLIF(
                COUNT(*) FILTER (
                    WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
                ),
                0
            ) AS available_rate_last_30d,
        AVG(minimum_nights) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        ) AS avg_minimum_nights_calendar_last_30d,
        AVG(maximum_nights) FILTER (
            WHERE date > CAST(:cutoff_date AS date) - INTERVAL '30 days'
        ) AS avg_maximum_nights_calendar_last_30d
    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    },
)

# convert numeric columns.
calendar_numeric_cols = [
    "available_days_last_90d",
    "available_rate_last_90d",
    "avg_minimum_nights_calendar_last_90d",
    "avg_maximum_nights_calendar_last_90d",
    "available_days_last_30d",
    "available_rate_last_30d",
    "avg_minimum_nights_calendar_last_30d",
    "avg_maximum_nights_calendar_last_30d",
]

for col in calendar_numeric_cols:
    calendar_features_all[col] = pd.to_numeric(calendar_features_all[col], errors="coerce")

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

(10480, 9)


,listing_id,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d
0,1443670960781261954,91,1.000000,2.0,30.0,30,1.0,2.0,30.0
1,896043282611946316,0,0.000000,5.0,25.0,0,0.0,5.0,25.0
2,958726726744532841,0,0.000000,2.0,7.0,0,0.0,2.0,7.0
3,39969190,0,0.000000,3.0,9.0,0,0.0,3.0,9.0
4,1272264495001498383,88,0.967033,2.0,365.0,27,0.9,2.0,365.0


## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [25]:
# write SQL to build one label row per listing.
# Use only dates after cutoff_date and up to label_end_date.
label_df = read_sql(
    """
    SELECT
        listing_id,
        COUNT(*) AS future_calendar_days_observed_30d,
        COUNT(*) FILTER (WHERE available IS TRUE) AS future_available_days_30d,
        COUNT(*) FILTER (WHERE available IS TRUE)::double precision
            / NULLIF(COUNT(*), 0) AS future_available_rate_30d,
        CASE
            WHEN COUNT(*) FILTER (WHERE available IS TRUE)::double precision
                / NULLIF(COUNT(*), 0) <= :threshold
            THEN 1
            ELSE 0
        END AS high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

# convert numeric columns and make high_demand_proxy integer.
label_numeric_cols = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
]

for col in label_numeric_cols:
    label_df[col] = pd.to_numeric(label_df[col], errors="coerce")

label_df["high_demand_proxy"] = label_df["high_demand_proxy"].astype(int)

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

(10480, 5)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy
0,1443670960781261954,30,30,1.0,0
1,896043282611946316,30,0,0.0,1
2,958726726744532841,30,0,0.0,1
3,39969190,30,0,0.0,1
4,1272264495001498383,30,30,1.0,0


In [26]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [27]:
# join static_features, review_features, calendar_features_all, and label_df.
feature_df = (
    label_df
    .merge(static_features, on="listing_id", how="inner")
    .merge(calendar_features_all, on="listing_id", how="inner")
    .merge(review_features, on="listing_id", how="left")
)

# add cutoff_date and dataset_version columns.
# fill missing review count features with zero.
# handle missing days_since_last_review for listings with no reviews.
feature_df["cutoff_date"] = cutoff_date
feature_df["dataset_version"] = DATASET_VERSION

review_count_cols = [
    "total_reviews_before_cutoff",
    "unique_reviewers_before_cutoff",
]
feature_df[review_count_cols] = feature_df[review_count_cols].fillna(0)

feature_df["days_since_last_review"] = feature_df["days_since_last_review"].fillna(PAST_WINDOW_DAYS)

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()

(10480, 33)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,room_type,property_type,accommodates,bedrooms,beds,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review,cutoff_date,dataset_version
0,1443670960781261954,30,30,1.0,0,Entire home/apt,Entire rental unit,2,1.0,1.0,...,1.0,2.0,30.0,5.0,5.0,371.200000,501.0,365.0,2026-08-11,v1_student
1,896043282611946316,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,5.0,25.0,2.0,2.0,469.000000,917.0,668.0,2026-08-11,v1_student
2,958726726744532841,30,0,0.0,1,Entire home/apt,Entire condo,2,1.0,NaN,...,0.0,2.0,7.0,23.0,23.0,277.304348,993.0,347.0,2026-08-11,v1_student
3,39969190,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,3.0,9.0,10.0,10.0,182.500000,423.0,546.0,2026-08-11,v1_student
4,1272264495001498383,30,30,1.0,0,Private room,Private room in townhouse,2,1.0,2.0,...,0.9,2.0,365.0,2.0,2.0,118.000000,201.0,557.0,2026-08-11,v1_student


## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [ ]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

missing_rates = feature_df.isna().mean()

high_missing_columns = [
    col
    for col in feature_df.columns
    if col not in protected_columns
    and missing_rates[col] > HIGH_MISSING_DROP_THRESHOLD
]

# compute missing rates.
# find high-missing columns outside protected columns.
# find constant columns outside protected columns.
# drop them from feature_df.
constant_columns = [
    col
    for col in feature_df.columns
    if col not in protected_columns
    and feature_df[col].nunique(dropna=False) <= 1
]

columns_to_drop = sorted(set(high_missing_columns) | set(constant_columns))

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

Columns to drop: []
New shape: (10480, 33)


## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [29]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

# add asserts for each validation rule.
assert duplicate_count == 0
assert "high_demand_proxy" in feature_df.columns
assert set(unique_target_values) == {0, 1}
assert missing_target_count == 0
assert len(present_forbidden_columns) == 0
assert len(future_leakage_columns) == 0

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))

duplicate_count: 0
missing_target_count: 0
unique_target_values: [0, 1]
present_forbidden_columns: []
future_leakage_columns: []
model_input_column_count: 26


In [30]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

,column,missing_rate
0,listing_price,0.439504
1,beds,0.436641
2,max_comment_len_before_cutoff,0.104676
3,avg_comment_len_before_cutoff,0.104676
4,bedrooms,0.029198
5,is_superhost,0.010973
6,bathrooms,0.001145
7,avg_maximum_nights_calendar_last_30d,0.000000
8,avg_maximum_nights_calendar_last_90d,0.000000
9,available_days_last_30d,0.000000


,high_demand_proxy,count,percentage
0,1,7994,0.7628
1,0,2486,0.2372


,index,future_calendar_days_observed_30d
0,count,10480.0
1,mean,30.0
2,std,0.0
3,min,30.0
4,25%,30.0
5,50%,30.0
6,75%,30.0
7,max,30.0


## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [31]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

# build metadata dictionary.
metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    # add cutoff, windows, source tables, target definition, exclusion rules
    "cutoff_date": str(cutoff_date),
    "history_start_date": str(history_start_date),
    "label_end_date": str(label_end_date),
    "past_window_days": PAST_WINDOW_DAYS,
    "future_window_days": FUTURE_WINDOW_DAYS,
    "high_demand_available_rate_threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    "source_tables": [
        "core.listing",
        "core.host",
        "core.neighbourhood",
        "core.review",
        "core.calendar_day",
    ],
    "target_definition": {
        "target_column": "high_demand_proxy",
        "rule": "high_demand_proxy = 1 if future_available_rate_30d <= threshold else 0",
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
    "exclusion_rules": {
        "forbidden_pii_columns": sorted(forbidden_columns),
        "dropped_feature_columns": columns_to_drop,
        "high_missing_drop_threshold": HIGH_MISSING_DROP_THRESHOLD,
    },
    "row_count": int(len(feature_df)),
    "column_count": int(feature_df.shape[1]),
    "model_input_column_count": int(len(model_input_columns)),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# build validation_report dictionary.
validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    # add missing_report, label_distribution, calendar_coverage_summary
    "missing_report": missing_report.to_dict(orient="records"),
    "label_distribution": label_distribution.to_dict(orient="records"),
    "calendar_coverage_summary": calendar_coverage_summary.to_dict(orient="records"),
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

Saved CSV: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features/listing_availability_features_v1_student.csv
Saved Parquet: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features/listing_availability_features_v1_student.parquet
Saved metadata: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features/listing_availability_features_v1_student_metadata.json
Saved validation report: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features/listing_availability_features_v1_student_validation_report.json
Saved PII audit: /home/samin/bootcamps/MLOps-Course/HW02/HW02_A/data/features/pii_audit_v1_student.csv


## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [32]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()

Final shape: (10480, 33)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,room_type,property_type,accommodates,bedrooms,beds,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review,cutoff_date,dataset_version
0,1443670960781261954,30,30,1.0,0,Entire home/apt,Entire rental unit,2,1.0,1.0,...,1.0,2.0,30.0,5.0,5.0,371.200000,501.0,365.0,2026-08-11,v1_student
1,896043282611946316,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,5.0,25.0,2.0,2.0,469.000000,917.0,668.0,2026-08-11,v1_student
2,958726726744532841,30,0,0.0,1,Entire home/apt,Entire condo,2,1.0,NaN,...,0.0,2.0,7.0,23.0,23.0,277.304348,993.0,347.0,2026-08-11,v1_student
3,39969190,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,3.0,9.0,10.0,10.0,182.500000,423.0,546.0,2026-08-11,v1_student
4,1272264495001498383,30,30,1.0,0,Private room,Private room in townhouse,2,1.0,2.0,...,0.9,2.0,365.0,2.0,2.0,118.000000,201.0,557.0,2026-08-11,v1_student


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10480 entries, 0 to 10479
Data columns (total 33 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   listing_id                            10480 non-null  int64  
 1   future_calendar_days_observed_30d     10480 non-null  int64  
 2   future_available_days_30d             10480 non-null  int64  
 3   future_available_rate_30d             10480 non-null  float64
 4   high_demand_proxy                     10480 non-null  int64  
 5   room_type                             10480 non-null  object 
 6   property_type                         10480 non-null  object 
 7   accommodates                          10480 non-null  int64  
 8   bedrooms                              10174 non-null  float64
 9   beds                                  5904 non-null   float64
 10  bathrooms                             10468 non-null  float64
 11  listing_price  